# Chapter 2 — Async Batch Processing for Intent Scoring

**Stage:** Foundations | **Book:** *Mastering Agentic AI for Customer Journey Marketing*

---

## What You Will Learn

| Concept | Why It Matters |
|---|---|
| Async LLM calls | Process hundreds of contacts without sequential bottlenecks |
| `asyncio` + `httpx` | Production-grade pattern for parallel HTTP requests |
| Semaphore-based concurrency | Respect API rate limits while maximizing throughput |
| Throughput metrics | Measure contacts/sec, latency percentiles, and error rates |

This notebook scales the intent-scoring pipeline from Notebook 01 to **500 contacts**,
classifying each contact's journey stage and recommending a next-best action — all
processed in parallel batches.

In [ ]:
# File      : 02_async_batch_processing.ipynb
# Stage     : Foundations
# Chapter   : Chapter 2 - Async Batch Processing
# Framework : asyncio, httpx
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os
import json
import time
import random
import asyncio
from dataclasses import dataclass, field
from typing import Any

import httpx
import pandas as pd

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"
print(f"USE_MOCK = {USE_MOCK}")

## 1 — Generate 500 Synthetic Contacts

Each contact has a randomized set of journey events so the LLM classifies
them into different stages.

In [ ]:
random.seed(42)

COMPANIES = ["acme", "globex", "initech", "hooli", "piedpiper", "waystar",
             "sterling", "genco", "oceanic", "umbrella", "aperture", "cyberdyne"]
FIRST_NAMES = ["alex", "jordan", "casey", "taylor", "morgan", "riley",
               "drew", "quinn", "avery", "cameron", "blake", "sage"]

STAGE_EVENT_TEMPLATES = {
    "awareness": [
        {"type": "page_view", "metadata": {"url": "/blog/ai-marketing"}},
        {"type": "ad_click", "metadata": {"campaign": "brand_awareness_q1"}},
    ],
    "consideration": [
        {"type": "page_view", "metadata": {"url": "/pricing"}},
        {"type": "email_open", "metadata": {"campaign": "nurture_series"}},
        {"type": "page_view", "metadata": {"url": "/case-studies"}},
    ],
    "decision": [
        {"type": "form_fill", "metadata": {"form": "demo_request"}},
        {"type": "email_click", "metadata": {"link": "/book-meeting"}},
    ],
    "onboarding": [
        {"type": "page_view", "metadata": {"url": "/docs/getting-started"}},
        {"type": "feature_use", "metadata": {"feature": "setup_wizard"}},
    ],
    "retention": [
        {"type": "feature_use", "metadata": {"feature": "journey_builder"}},
        {"type": "support_ticket", "metadata": {"topic": "integration_help"}},
    ],
    "advocacy": [
        {"type": "nps_response", "metadata": {"score": 9}},
        {"type": "referral", "metadata": {"referred": "newco@example.com"}},
    ],
}

STAGE_NAMES = list(STAGE_EVENT_TEMPLATES.keys())


def generate_contact(contact_id: int) -> dict:
    """Generate a synthetic contact with random journey events."""
    company = random.choice(COMPANIES)
    name = random.choice(FIRST_NAMES)
    email = f"{name}{contact_id}@{company}.com"

    # Pick a random "current" stage — include all events up to that stage
    stage_idx = random.randint(0, len(STAGE_NAMES) - 1)
    events = []
    for s in STAGE_NAMES[: stage_idx + 1]:
        for template in STAGE_EVENT_TEMPLATES[s]:
            events.append({**template, "contact_email": email})

    return {
        "contact_id": contact_id,
        "email": email,
        "company": company,
        "expected_stage": STAGE_NAMES[stage_idx],
        "events": events,
    }


NUM_CONTACTS = 500
contacts = [generate_contact(i) for i in range(NUM_CONTACTS)]

# Distribution summary
stage_counts = {}
for c in contacts:
    stage_counts[c["expected_stage"]] = stage_counts.get(c["expected_stage"], 0) + 1

print(f"Generated {NUM_CONTACTS} contacts")
print("Stage distribution:")
for stage in STAGE_NAMES:
    print(f"  {stage:<15} {stage_counts.get(stage, 0):>4}")

## 2 — Intent Scoring Pipeline

The core classification function — wraps the LLM call (or mock) and returns
a structured result for each contact.

In [ ]:
SYSTEM_PROMPT = """You are a B2B SaaS marketing analyst AI. Given a contact's journey events,
classify their current journey stage and recommend a next-best action.

Stages: Awareness, Consideration, Decision, Onboarding, Retention, Advocacy.

Return ONLY valid JSON:
{"stage": "<stage>", "confidence": <float>, "next_action": "<action>"}"""

NEXT_ACTIONS = {
    "awareness":     "Send educational content series",
    "consideration": "Trigger case study email + retarget ads",
    "decision":      "Assign SDR for personalized outreach",
    "onboarding":    "Send onboarding checklist + schedule CSM call",
    "retention":     "Offer advanced training webinar",
    "advocacy":      "Invite to referral program + request G2 review",
}


async def score_contact_mock(contact: dict) -> dict:
    """Mock scoring — simulates LLM latency and returns deterministic results."""
    await asyncio.sleep(random.uniform(0.01, 0.05))  # simulate 10-50ms latency
    stage = contact["expected_stage"]
    return {
        "contact_id": contact["contact_id"],
        "email": contact["email"],
        "stage": stage.capitalize(),
        "confidence": round(random.uniform(0.75, 0.98), 2),
        "next_action": NEXT_ACTIONS[stage],
    }


async def score_contact_live(contact: dict, client: httpx.AsyncClient,
                             semaphore: asyncio.Semaphore) -> dict:
    """Score a single contact via OpenAI API with concurrency limiting."""
    async with semaphore:
        payload = {
            "model": "gpt-4.1",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(contact["events"])},
            ],
            "temperature": 0.2,
            "response_format": {"type": "json_object"},
        }
        headers = {
            "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
            "Content-Type": "application/json",
        }
        resp = await client.post(
            "https://api.openai.com/v1/chat/completions",
            json=payload,
            headers=headers,
            timeout=30.0,
        )
        resp.raise_for_status()
        result = json.loads(resp.json()["choices"][0]["message"]["content"])
        result["contact_id"] = contact["contact_id"]
        result["email"] = contact["email"]
        return result


print("Pipeline functions defined.")

## 3 — Async Batch Processor with Metrics

We process all 500 contacts concurrently (bounded by a semaphore) and
collect detailed timing metrics.

In [ ]:
@dataclass
class BatchMetrics:
    """Collects throughput and latency metrics for the batch run."""
    total_contacts: int = 0
    successful: int = 0
    failed: int = 0
    start_time: float = 0.0
    end_time: float = 0.0
    latencies: list = field(default_factory=list)

    @property
    def elapsed_sec(self) -> float:
        return self.end_time - self.start_time

    @property
    def throughput(self) -> float:
        return self.successful / self.elapsed_sec if self.elapsed_sec > 0 else 0

    def percentile(self, p: float) -> float:
        if not self.latencies:
            return 0.0
        sorted_lat = sorted(self.latencies)
        idx = int(len(sorted_lat) * p / 100)
        return sorted_lat[min(idx, len(sorted_lat) - 1)]

    def summary(self) -> str:
        return (
            f"\n{'='*50}\n"
            f"  BATCH PROCESSING METRICS\n"
            f"{'='*50}\n"
            f"  Total contacts:      {self.total_contacts}\n"
            f"  Successful:          {self.successful}\n"
            f"  Failed:              {self.failed}\n"
            f"  Wall-clock time:     {self.elapsed_sec:.2f}s\n"
            f"  Throughput:          {self.throughput:.1f} contacts/sec\n"
            f"  Latency p50:         {self.percentile(50)*1000:.1f}ms\n"
            f"  Latency p95:         {self.percentile(95)*1000:.1f}ms\n"
            f"  Latency p99:         {self.percentile(99)*1000:.1f}ms\n"
            f"{'='*50}"
        )


print("BatchMetrics class defined.")

In [ ]:
MAX_CONCURRENCY = 20  # max simultaneous API calls


async def process_batch(contacts: list[dict]) -> tuple[list[dict], BatchMetrics]:
    """Process all contacts concurrently and return results + metrics."""
    metrics = BatchMetrics(total_contacts=len(contacts))
    results = []
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    async def _score_one(contact: dict):
        t0 = time.monotonic()
        try:
            if USE_MOCK:
                result = await score_contact_mock(contact)
            else:
                async with httpx.AsyncClient() as client:
                    result = await score_contact_live(contact, client, semaphore)
            elapsed = time.monotonic() - t0
            metrics.latencies.append(elapsed)
            metrics.successful += 1
            results.append(result)
        except Exception as e:
            metrics.failed += 1
            results.append({
                "contact_id": contact["contact_id"],
                "email": contact["email"],
                "error": str(e),
            })

    metrics.start_time = time.monotonic()
    tasks = [asyncio.create_task(_score_one(c)) for c in contacts]
    await asyncio.gather(*tasks)
    metrics.end_time = time.monotonic()

    return results, metrics


print(f"Batch processor ready — max concurrency: {MAX_CONCURRENCY}")

## 4 — Run the Batch

In [ ]:
results, metrics = await process_batch(contacts)
print(metrics.summary())

## 5 — Results Analysis

In [ ]:
df = pd.DataFrame([r for r in results if "error" not in r])

print("\n=== Stage Distribution ===")
stage_dist = df["stage"].value_counts().sort_index()
for stage, count in stage_dist.items():
    bar = "█" * (count // 5)
    print(f"  {stage:<15} {count:>4}  {bar}")

print(f"\n=== Confidence Statistics ===")
print(f"  Mean:   {df['confidence'].mean():.3f}")
print(f"  Median: {df['confidence'].median():.3f}")
print(f"  Min:    {df['confidence'].min():.3f}")
print(f"  Max:    {df['confidence'].max():.3f}")

print("\n=== Sample Results (first 10) ===")
print(df[["contact_id", "email", "stage", "confidence", "next_action"]].head(10).to_string(index=False))

## 6 — Next-Best-Action Summary by Stage

In [ ]:
print("\n=== Next-Best-Action Queue ===")
print(f"{'Stage':<15} {'Count':>6}  {'Action'}")
print("-" * 70)

for stage in ["Awareness", "Consideration", "Decision", "Onboarding", "Retention", "Advocacy"]:
    stage_df = df[df["stage"] == stage]
    if len(stage_df) > 0:
        action = stage_df["next_action"].iloc[0]
        print(f"  {stage:<15} {len(stage_df):>4}  {action}")

## 7 — Throughput Comparison: Sequential vs Async

In [ ]:
# Simulate sequential processing time for comparison
avg_latency = sum(metrics.latencies) / len(metrics.latencies) if metrics.latencies else 0.03
sequential_estimate = avg_latency * metrics.total_contacts

print("\n=== Throughput Comparison ===")
print(f"{'Approach':<25} {'Time (s)':>10} {'Contacts/sec':>15} {'Speedup':>10}")
print("-" * 62)
print(f"{'Sequential (estimated)':<25} {sequential_estimate:>10.2f} {metrics.total_contacts/sequential_estimate:>15.1f} {'1.0x':>10}")
print(f"{'Async (actual)':<25} {metrics.elapsed_sec:>10.2f} {metrics.throughput:>15.1f} {sequential_estimate/metrics.elapsed_sec:>9.1f}x")

# With live APIs, typical improvements:
print("\n--- Typical live-API benchmarks ---")
print("  Sequential (GPT-4.1):  ~2-3 contacts/sec")
print("  Async (20 concurrent): ~30-40 contacts/sec")
print("  Speedup:               ~10-15x")

---

## Key Takeaways

1. **Async processing is essential for production martech** — sequential LLM calls cannot
   scale to real contact databases (10K-1M+ records).

2. **`asyncio` + `httpx` is the recommended pattern** — it gives fine-grained control over
   concurrency via semaphores, which is critical for respecting API rate limits.

3. **Structured metrics matter** — track throughput, latency percentiles, and error rates
   to right-size your concurrency and detect API degradation.

4. **Each contact gets a personalized next-best-action** — the LLM does not just classify;
   it recommends, turning raw signals into actionable marketing plays.

**Next:** [03_agentic_patterns_by_stage.ipynb](./03_agentic_patterns_by_stage.ipynb) — learn
the six agentic design patterns, each mapped to a customer journey stage.